# 02 — Introduction to Embeddings

**Goal**: Understand what embeddings are and how they capture meaning.

An **embedding** is a vector (list of numbers) that represents the "meaning" of a word/sentence.
Similar meanings → similar vectors.

```
"king" → [0.2, 0.5, -0.1, ...]  (close to "queen")
"apple" → [0.8, 0.1, 0.3, ...] (close to "fruit")
```

In [ ]:
# Install if needed
# !pip install sentence-transformers scikit-learn numpy

import numpy as np  # Numerical computing: arrays, matrix ops, linear algebra
from sentence_transformers import SentenceTransformer  # Generate sentence embeddings locally
from sklearn.metrics.pairwise import cosine_similarity  # Compute similarity between embedding vectors

print("Imports OK")

## 1. Why Embeddings?

Models can't understand raw text — they need numbers.

**Bad approach** (BoW from previous notebook):
- `"dog"` → `[0, 0, 1, 0, ...]` (one-hot, no meaning)
- `"cat"` → `[0, 1, 0, 0, ...]` (just as different from "dog" as from "car")

**Good approach** (embeddings):
- `"dog"` → close to `"cat"` (both animals)
- `"dog"` → far from `"car"`

Embeddings capture **semantic similarity**.

In [ ]:
# Load a free, local sentence embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 2. Embedding Words

In [ ]:
words = ["king", "queen", "man", "woman", "apple", "orange", "car", "bicycle"]

# Get embeddings
word_embeddings = model.encode(words)

print(f"Shape: {word_embeddings.shape}")  # (8 words, 384 dimensions)
print(f"\nEmbedding for 'king':")
print(word_embeddings[0][:10])  # First 10 numbers
print(f"... (384 numbers total)")

## 3. Visualizing Similarity

Each word is a point in 384-dimensional space. We can't visualize that, but we can compute **distances**.

In [ ]:
# Compute cosine similarity between every pair
similarity_matrix = cosine_similarity(word_embeddings)

# Display as a heatmap-like table
print(f"{'':>10}", end="")
for w in words:
    print(f"{w:>10}", end="")
print()
for i, w1 in enumerate(words):
    print(f"{w1:>10}", end="")
    for j in range(len(words)):
        print(f"{similarity_matrix[i,j]:>8.3f}  ", end="")
    print()

**Observations**:
- king ↔ queen: high similarity (both royalty)
- apple ↔ orange: high similarity (both fruits)
- king ↔ apple: low similarity
- car ↔ bicycle: high similarity (both vehicles)

This is the **semantic** understanding that makes embeddings powerful.

## 4. The Classic Analogy: king - man + woman = queen

Embeddings capture relationships as vector arithmetic!

In [ ]:
# Get embeddings for the analogy
emb_king = model.encode("king")
emb_man = model.encode("man")
emb_woman = model.encode("woman")

# king - man + woman
result_vector = emb_king - emb_man + emb_woman

# Find closest word in our vocabulary
all_words = words + ["queen", "prince", "princess", "apple"]
all_embs = model.encode(all_words)

similarities = cosine_similarity([result_vector], all_embs)[0]
most_similar_idx = np.argsort(similarities)[::-1]

print("king - man + woman = ?")
print(f"{'Word':<10} {'Similarity':<10}")
print("-" * 20)
for idx in most_similar_idx[:3]:
    print(f"{all_words[idx]:<10} {similarities[idx]:.3f}")

## 5. Sentence Embeddings

We can also embed entire sentences. This is what RAG systems use!

In [ ]:
sentences = [
    "I love machine learning",
    "Deep learning is fascinating",
    "I enjoy hiking in the mountains",
    "The weather today is sunny",
    "Neural networks are powerful"
]

sent_embs = model.encode(sentences)
sent_sim = cosine_similarity(sent_embs)

print("Sentence similarity matrix:")
for i, s1 in enumerate(sentences):
    print(f"\n{s1}")
    for j, s2 in enumerate(sentences):
        if i < j:
            print(f"  ↔ {s2:<45} {sent_sim[i,j]:.3f}")

## 6. Using Ollama for Embeddings (Alternative)

You can also get embeddings from your local Ollama models. This keeps everything local.

In [ ]:
import requests  # HTTP client to call local Ollama API for embeddings

# Get embedding from Ollama's nomic-embed-text model
def get_ollama_embedding(text):
    resp = requests.post("http://localhost:11434/api/embeddings", json={
        "model": "nomic-embed-text",
        "prompt": text
    })
    return resp.json()["embedding"]

# Test it
emb = get_ollama_embedding("Hello world")
print(f"Ollama embedding dimension: {len(emb)}")
print(f"First 10 values: {emb[:10]}")

## Key Takeaways

1. **Embeddings** = numbers that represent meaning
2. **Similar meaning** → similar vectors (close in vector space)
3. **Semantic relationships** are captured (king - man + woman ≈ queen)
4. **Sentence embeddings** power RAG (retrieve relevant docs by meaning)
5. You have **two options**: `sentence-transformers` (fast, local) or Ollama embeddings (consistent with your LLM)

**Next**: Use embeddings to search and find similar documents →